# 🌼 Flower Classification – End-to-End ML Pipeline
**Dataset:** TF Flowers (≈3,670 real photos, 5 classes). **Model:** MobileNetV2 transfer learning.

Sections: 1 Data acquisition · 2 EDA & visualisations · 3 Preprocessing · 4 Baseline CNN · 5 Optimised model · 6 Evaluation · 7 Save model & prediction function · 8 Retraining demo

> Run top-to-bottom (Kernel → Restart & Run All) and **commit the notebook with outputs**.

In [ ]:
import os, sys, json, random
from pathlib import Path
# In Colab, first upload/extract the repository ZIP under /content or clone it there.
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/flower_mlops')]
if Path('/content').exists(): candidates += [p.parent.parent for p in Path('/content').rglob('config.py') if p.parent.name == 'src']
ROOT = next((p.resolve() for p in candidates if (p / 'src' / 'config.py').exists()), None)
if ROOT is None: raise FileNotFoundError('Project source not found. In Colab, upload/extract the flower_mlops repository ZIP under /content, then rerun.')
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report

from src import config as C, preprocessing as P, model as M
from src.insights import compute_insights

tf.keras.utils.set_random_seed(C.SEED)
print("TensorFlow", tf.__version__)

## 1. Data acquisition
The dataset is downloaded from the TensorFlow public bucket and split **80/20 per class** into `data/train` and `data/test` (the test set is never used for training or tuning).

In [ ]:
P.download_and_split()
class_names = P.get_class_names(C.TRAIN_DIR)
print("Classes:", class_names)
for split in (C.TRAIN_DIR, C.TEST_DIR):
    print(split.name, {c: len(list((split/c).glob('*'))) for c in class_names})

In [ ]:
# A look at the raw data
fig, axes = plt.subplots(len(class_names), 4, figsize=(10, 2.4*len(class_names)))
for i, c in enumerate(class_names):
    files = random.sample(list((C.TRAIN_DIR/c).glob('*.jpg')), 4)
    for j, f in enumerate(files):
        axes[i, j].imshow(plt.imread(f)); axes[i, j].axis('off')
        if j == 0: axes[i, j].set_title(c, loc='left')
plt.tight_layout(); plt.show()

## 2. Exploratory data analysis (3+ features, with interpretation)

In [ ]:
ins = compute_insights(per_class_sample=100)

In [ ]:
# Feature 1: class distribution
counts = pd.Series(ins['class_counts']['train']).sort_values()
ax = counts.plot.barh(figsize=(6,3), color='steelblue'); ax.set_xlabel('training images')
for i, v in enumerate(counts): ax.text(v+5, i, str(v), va='center')
plt.title('Class distribution (train)'); plt.show()
print(f"Imbalance ratio largest/smallest: {counts.max()/counts.min():.2f}")

**Interpretation 1 – Class distribution.** The classes are *mildly* imbalanced (largest/smallest ≈ 1.4×, see printed ratio). Not severe enough to need resampling, but it means accuracy alone can hide weak performance on the smaller classes, so we report **per-class precision/recall/F1** and a confusion matrix.

In [ ]:
# Feature 2: brightness per class
plt.figure(figsize=(7,3.5))
plt.boxplot([ins['brightness'][c] for c in class_names], labels=class_names)
plt.ylabel('mean pixel intensity'); plt.title('Brightness per class'); plt.show()
print(pd.Series({c: np.median(v) for c, v in ins['brightness'].items()}).round(1))

**Interpretation 2 – Brightness.** *(Read the plot and complete this sentence with your numbers, e.g. which class is brightest/darkest and how wide the spread is.)* Overlapping boxes mean brightness alone cannot discriminate classes, and wide spreads (mixed sun/shade photos) justify the contrast augmentation used in preprocessing.

In [ ]:
# Feature 3: average colour profile
rgb = pd.DataFrame(ins['rgb_mean'], index=['R','G','B']).T
rgb.plot.bar(color=['tab:red','tab:green','tab:blue'], figsize=(7,3.5)); plt.ylabel('mean channel value')
plt.title('Average colour profile per class'); plt.show()
print((rgb['R']/rgb.sum(axis=1)).round(3).sort_values())

**Interpretation 3 – Colour.** All classes are green-dominant (leaves/stems in the background). The class with the highest red share is the most “colour-distinctive”; classes with similar profiles are the ones we expect to be confused (check the confusion matrix later). Colour helps, but shape/texture features learned by the CNN are needed.

In [ ]:
# Feature 4: image dimensions
rows = [(c, w, h, w/h) for c, s in ins['size'].items() for w, h in s]
sz = pd.DataFrame(rows, columns=['class','w','h','aspect'])
sns.boxplot(data=sz, x='class', y='aspect'); plt.title('Aspect ratio (w/h) per class'); plt.show()
print(sz[['w','h','aspect']].describe().round(2))

**Interpretation 4 – Image size.** Images are not uniform and mostly not square, so every image must be resized to a fixed 224×224 (MobileNetV2 input). This stretches some images slightly; centre-crop/padding is a possible improvement.

## 3. Data preprocessing
1. Stratified **85/15 train/validation split** of `data/train` (test set stays untouched)
2. Decode → **resize to 224×224** → batch(32) with `tf.data`
3. **Augmentation (training only):** horizontal flip, rotation ±10 %, zoom ±10 %, contrast ±10 %
4. **Normalisation:** pixels scaled to [-1, 1] by a `Rescaling` layer *inside* the model, so the API receives raw pixels and cannot diverge from training

In [ ]:
paths, labels = P.list_files(C.TRAIN_DIR, class_names)
(tp, tl), (vp, vl) = P.stratified_split(paths, labels, 0.15)
n = len(class_names)
train_ds = P.paths_to_dataset(tp, tl, n, training=True)
val_ds   = P.paths_to_dataset(vp, vl, n)
print(f"train={len(tp)}  val={len(vp)}")

# show augmentation on one image
x, y = next(iter(train_ds))
plt.figure(figsize=(10,2.5))
for i in range(5):
    plt.subplot(1,5,i+1); plt.imshow(x[i].numpy().astype('uint8')); plt.title(class_names[y[i].numpy().argmax()]); plt.axis('off')
plt.show()

## 4. Baseline: small CNN from scratch (no pre-training)

In [ ]:
from tensorflow.keras import layers
baseline = tf.keras.Sequential([
    layers.Input((*C.IMG_SIZE, 3)), layers.Rescaling(1/255),
    layers.Conv2D(32,3,activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64,3,activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(128,3,activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(128, activation='relu'), layers.Dense(n, activation='softmax')
], name='baseline_cnn')
M.compile_model(baseline, 1e-3)
h_base = baseline.fit(train_ds, validation_data=val_ds, epochs=10, verbose=2,
                      callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

## 5. Optimised model: MobileNetV2 transfer learning
Optimisation techniques used: **pre-trained model**, **L2 regularisation**, **Dropout**, **Adam optimiser** with a tuned learning rate, **EarlyStopping**, **ReduceLROnPlateau**, **two-phase fine-tuning**, plus **data augmentation** and a small **hyper-parameter search**.
### 5a. Hyper-parameter search (head learning rate × dropout, 3 epochs each)

In [ ]:
results = []
for lr in (1e-3, 3e-4):
    for dr in (0.2, 0.4):
        m = M.build_model(n, dropout=dr); M.compile_model(m, lr)
        h = m.fit(train_ds, validation_data=val_ds, epochs=3, verbose=0)
        results.append({'lr': lr, 'dropout': dr, 'val_acc': max(h.history['val_accuracy']), 'val_loss': min(h.history['val_loss'])})
tune = pd.DataFrame(results).sort_values('val_acc', ascending=False); display(tune)
best = tune.iloc[0]; print('Best:', dict(best))

### 5b. Phase 1 – train the head (base frozen) with the best settings

In [ ]:
model = M.build_model(n, dropout=float(best.dropout)); M.compile_model(model, float(best.lr))
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=M.make_callbacks(), verbose=2)

### 5c. Phase 2 – fine-tune the top 30 MobileNetV2 layers at a low learning rate

In [ ]:
M.unfreeze_top(model, 30); M.compile_model(model, 1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=M.make_callbacks(), verbose=2)

In [ ]:
def curves(histories, title):
    acc = sum([h.history['accuracy'] for h in histories], []);     vacc = sum([h.history['val_accuracy'] for h in histories], [])
    loss = sum([h.history['loss'] for h in histories], []);        vloss = sum([h.history['val_loss'] for h in histories], [])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(acc, label='train'); ax[0].plot(vacc, label='val'); ax[0].set_title(title+' – accuracy'); ax[0].legend()
    ax[1].plot(loss, label='train'); ax[1].plot(vloss, label='val'); ax[1].set_title(title+' – loss'); ax[1].legend()
    plt.show()
curves([h_base], 'Baseline CNN'); curves([h1, h2], 'MobileNetV2')

## 6. Evaluation on the held-out test set (accuracy, loss, precision, recall, F1, confusion matrix)

In [ ]:
m_base, _, _ = M.evaluate_model(baseline, class_names)
m_opt, y_true, y_pred = M.evaluate_model(model, class_names)
keys = ['accuracy','loss','precision','recall','f1']
cmp = pd.DataFrame({'Baseline CNN': {k: m_base[k] for k in keys}, 'MobileNetV2 (optimised)': {k: m_opt[k] for k in keys}}).round(4)
display(cmp)

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
cm = np.array(m_opt['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix – test set'); plt.show()

**Discussion.** *(Complete with your numbers.)* The pre-trained, regularised model should clearly beat the scratch CNN because ImageNet features transfer well to flowers and the dataset is small. Use the confusion matrix to name the most-confused pair of classes and connect it to the colour/brightness findings in Section 2.

## 7. Save the model + prediction function

In [ ]:
version = M.save_artifacts(model, class_names, m_opt, 'notebook_training')
print('Saved', C.MODEL_PATH, 'as version', version)

In [ ]:
from src.prediction import predict_image
test_files = [f for c in class_names for f in random.sample(list((C.TEST_DIR/c).glob('*.jpg')), 1)]
plt.figure(figsize=(12,3))
for i, f in enumerate(test_files):
    r = predict_image(f.read_bytes())
    plt.subplot(1, len(test_files), i+1); plt.imshow(plt.imread(f)); plt.axis('off')
    plt.title(f"true: {f.parent.name}\npred: {r['label']} ({r['confidence']:.0%})", fontsize=9)
plt.show()

## 8. Retraining demo
`M.retrain()` loads **our saved model** (`models/flower_model.h5`) as the pre-trained base, merges *uploaded* images (`data/uploads/`) with a replay sample of the original data, fine-tunes at a low learning rate, re-evaluates, and saves a new version. In production this is triggered from the UI/API (`POST /retrain`) or automatically at 50 pending uploads.

To demo it here, simulate an upload using a few images through the same preprocessing function the API uses:

In [ ]:
RUN_RETRAIN_DEMO = False   # set True to run
if RUN_RETRAIN_DEMO:
    for c in class_names:
        for f in random.sample(list((C.TRAIN_DIR/c).glob('*.jpg')), 10):
            P.save_uploaded_image(f.read_bytes(), f.name, c)
    meta = M.retrain(epochs=3)
    print(meta['latest']['metrics']['accuracy'], 'vs before', meta['latest']['accuracy_before'])